In [61]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

model_df = pd.read_csv('f1_2023_lap_model_data.csv')


C:\Users\bhavi\AppData\Local\Temp\ipykernel_21384\2051993027.py:7: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  model_df = pd.read_csv('f1_2023_lap_model_data.csv')


In [2]:
import pandas as pd

track_df = pd.read_csv('track_char.csv')

# Clean whitespace and casing before merging
track_df['EventName'] = track_df['EventName'].str.strip()
track_df['TrackDirection'] = track_df['TrackDirection'].str.strip().str.capitalize()

EmptyDataError: No columns to parse from file

In [62]:
model_df = model_df.dropna(subset=['LapTime_Seconds'])
print(model_df[model_df['LapTime_Seconds'].isna()][['Deleted', 'DeletedReason', 'IsAccurate']])
model_df = model_df.dropna(subset=['LapTime_Seconds', 'TyreLife', 'TrackTemp', 'AirTemp'])

Empty DataFrame
Columns: [Deleted, DeletedReason, IsAccurate]
Index: []


In [63]:
print(model_df[['LapTime_Seconds', 'TyreLife', 'TrackTemp', 'AirTemp']].describe())
print(model_df['Compound'].value_counts())
print(model_df.isna().sum())

       LapTime_Seconds      TyreLife     TrackTemp       AirTemp
count     20687.000000  20687.000000  20687.000000  20687.000000
mean         88.933756     14.860975     35.077522     24.486924
std          10.823729      9.774253      7.180337      4.347689
min          67.012000      1.000000     17.500000     15.700000
25%          79.753000      7.000000     30.300000     20.800000
50%          87.721000     13.000000     33.300000     25.500000
75%          97.700500     20.000000     40.700000     27.300000
max         148.490000     58.000000     50.200000     31.500000
Compound
HARD            10374
MEDIUM           7030
SOFT             2781
INTERMEDIATE      471
WET                31
Name: count, dtype: int64
Time                      0
Driver                    0
DriverNumber              0
LapTime                   0
LapNumber                 0
Stint                     0
PitOutTime            20687
PitInTime             20687
Sector1Time             181
Sector2Time       

In [76]:
# Sort races chronologically
sorted_events = model_df[['RoundNumber', 'EventName']].drop_duplicates().sort_values('RoundNumber')
event_order = sorted_events['EventName'].tolist()

# Hold out the last 4-5 races as test
n_test_races = 5
train_events = event_order[:-n_test_races]
test_events = event_order[-n_test_races:]

train_df = model_df[model_df['EventName'].isin(train_events)]
test_df = model_df[model_df['EventName'].isin(test_events)]

print(f"Train races: {len(train_events)}, Test races: {len(test_events)}")
print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")
print("Test races:", test_events)

Train races: 17, Test races: 5
Train rows: 15979, Test rows: 4708
Test races: ['United States Grand Prix', 'Mexico City Grand Prix', 'São Paulo Grand Prix', 'Las Vegas Grand Prix', 'Abu Dhabi Grand Prix']


In [77]:

features = ['TyreLife', 'Compound', 'FreshTyre', 'Stint', 'TrackTemp', 
            'AirTemp', 'EventName', 'Driver', 'LapNumber']

X_train = pd.get_dummies(train_df[features], columns=['Compound', 'EventName', 'Driver'])
X_test = pd.get_dummies(test_df[features], columns=['Compound', 'EventName', 'Driver'])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df['LapTime_Seconds']
y_test = test_df['LapTime_Seconds']

In [81]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [82]:

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"MAE: {mae:.3f} seconds")

MAE: 9.222 seconds


In [85]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))

AirTemp                               0.348397
EventName_Belgian Grand Prix          0.218645
EventName_Azerbaijan Grand Prix       0.096254
EventName_British Grand Prix          0.080581
TrackTemp                             0.076715
EventName_Austrian Grand Prix         0.068395
EventName_Monaco Grand Prix           0.024219
LapNumber                             0.018692
EventName_Japanese Grand Prix         0.015792
EventName_Miami Grand Prix            0.010878
EventName_Qatar Grand Prix            0.007957
EventName_Italian Grand Prix          0.006141
TyreLife                              0.004949
EventName_Australian Grand Prix       0.004811
EventName_Saudi Arabian Grand Prix    0.004666
dtype: float64
